# 📚 Imports and Configurations

In [22]:
# =============================
# IMPORTS & CONFIG
# =============================
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

# =============================
# CONFIG
# =============================
DATA_DIR = "/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode"
TRAIN_CSV = f"{DATA_DIR}/Train/Train.csv"
TRAIN_IMG_DIR = f"{DATA_DIR}/Train/Image"

TEST_CSV = f"{DATA_DIR}/Test/Test.csv"
TEST_IMG_DIR = f"{DATA_DIR}/Test/Image"

BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-4
IMG_SIZE = 224
device = "cuda" if torch.cuda.is_available() else "cpu"

# 🗂 Dataset Class

In [23]:
# =============================
# DATASET CLASS
# =============================
class MemeDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, train=True, label2id=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.train = train
        self.label2id = label2id

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["Image_name"])
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        if self.train:
            label = self.label2id[row["Label"]]
            return img, label
        
        return img, row["Image_name"]

# 📊 Load Data

In [24]:
# =============================
# LOAD DATA
# =============================
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

classes = sorted(train_df["Label"].unique())
label2id = {c:i for i,c in enumerate(classes)}
id2label = {v:k for k,v in label2id.items()}


# 🎨 Data Transforms

In [25]:
# =============================
# TRANSFORMS
# =============================
train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 📥 Dataloaders

In [26]:
# =============================
# DATALOADERS
# =============================
train_ds = MemeDataset(train_df, TRAIN_IMG_DIR, train_tf, True, label2id)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

test_ds = MemeDataset(test_df, TEST_IMG_DIR, test_tf, False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# 🖼 Resnet18 

In [27]:
# =============================
# PRETRAINED MODEL
# =============================
model = timm.create_model('resnet18', pretrained=True, num_classes=len(classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)


# 🚀 Training Loop with TQDM

In [28]:

# =============================
# TRAINING LOOP
# =============================
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}")

print("Training finished!")

Epoch 1/50: 100%|██████████| 90/90 [00:58<00:00,  1.54it/s, loss=0.4491]


Epoch 1 completed. Average Loss: 0.5974


Epoch 2/50: 100%|██████████| 90/90 [00:56<00:00,  1.59it/s, loss=0.5790]


Epoch 2 completed. Average Loss: 0.5283


Epoch 3/50: 100%|██████████| 90/90 [00:56<00:00,  1.60it/s, loss=0.5078]


...

Epoch 48 completed. Average Loss: 0.0173


Epoch 49/50: 100%|██████████| 90/90 [00:55<00:00,  1.61it/s, loss=0.0086]


Epoch 49 completed. Average Loss: 0.0150


Epoch 50/50: 100%|██████████| 90/90 [00:55<00:00,  1.62it/s, loss=0.0116]

Epoch 50 completed. Average Loss: 0.0082
Training finished!


# 💾 Inference & Creating Submission

In [30]:
# =============================
# PREDICTION & SUBMISSION
# =============================
import pandas as pd
import torch

model.eval()
preds = []
image_names = []

with torch.no_grad():
    for imgs, names in tqdm(test_loader, desc="Predicting"):
        imgs = imgs.to(device)
        outputs = model(imgs)
        predicted = torch.argmax(outputs, dim=1).cpu().numpy()

        preds.extend([id2label[p] for p in predicted])
        image_names.extend(names)

# Create submission DataFrame
submission_df = pd.DataFrame({
    "Image_name": image_names,
    "Label": preds
})

# Save to CSV
submission_csv = "/kaggle/working/PoliMemeDecode_Submission.csv"
submission_df.to_csv(submission_csv, index=False)

print(f"Submission CSV saved at: {submission_csv}")
print(submission_df.head())


Predicting: 100%|██████████| 11/11 [00:06<00:00,  1.62it/s]

Submission CSV saved at: /kaggle/working/PoliMemeDecode_Submission.csv
     Image_name         Label
0  test0001.jpg  NonPolitical
1  test0002.jpg  NonPolitical
2  test0003.jpg  NonPolitical
3  test0004.jpg  NonPolitical
4  test0005.jpg  NonPolitical
